# Fertilizer Recommendation System - Big Data Methods with Spark MLlib

This notebook implements a big data approach for fertilizer recommendation using PySpark and MLlib.

## Setting up Spark Environment

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import pyspark.sql.functions as F

# Create Spark session
spark = SparkSession.builder.appName("FertilizerRecommendation").getOrCreate()

## Data Loading and Exploration

In [ ]:
# Load the dataset
df = spark.read.csv("Fertilizer Prediction.csv", header=True, inferSchema=True)

# Display schema
df.printSchema()

# Show first few rows
df.show(5)

# Basic statistics
df.describe().show()

# Count of records
print(f"Total records: {df.count()}")

# Check for null values
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

## Data Preprocessing

In [ ]:
# Index categorical columns
soil_indexer = StringIndexer(inputCol="Soil Type", outputCol="Soil_Type_Index")
crop_indexer = StringIndexer(inputCol="Crop Type", outputCol="Crop_Type_Index")
fert_indexer = StringIndexer(inputCol="Fertilizer Name", outputCol="label")

# Assemble features
assembler = VectorAssembler(
    inputCols=["Temparature", "Humidity", "Moisture", "Soil_Type_Index", "Crop_Type_Index", "Nitrogen", "Potassium", "Phosphorous"],
    outputCol="features"
)

# Scale features
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

# Split data
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

## Model Training with MLlib

In [ ]:
# Create Random Forest Classifier
rf = RandomForestClassifier(featuresCol="scaledFeatures", labelCol="label", numTrees=100, seed=42)

# Create pipeline
pipeline = Pipeline(stages=[soil_indexer, crop_indexer, fert_indexer, assembler, scaler, rf])

# Train the model
model = pipeline.fit(train_df)

## Model Evaluation

In [ ]:
# Make predictions
predictions = model.transform(test_df)

# Evaluate
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy}")

# F1 Score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1 = evaluator_f1.evaluate(predictions)
print(f"F1 Score: {f1}")

# Show predictions
predictions.select("Fertilizer Name", "prediction", "probability").show(10)

# Feature importance (if available)
# Note: RandomForest in Spark MLlib doesn't directly provide feature importances like sklearn
# We can use tree models or other methods

spark.stop()